# Chapter 14 Lab: Audio, Time, and Prediction-Time Realism (`ch09`)

In this lab, we explore time series, forecasting, and the pitfalls of naive evaluation. We'll see how temporal structure differs from iid data, why temporal leakage is dangerous, and how to build realistic models.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Seed for reproducibility
np.random.seed(42)

## 3. Spectrogram Basics

Generate synthetic audio: sum of two sine waves. Compute FFT and spectrogram to visualize frequency content over time.

In [ ]:
# Signal parameters
sample_rate = 8000  # Hz
duration = 1.0      # seconds
t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

# Generate signal: two sine waves (440 Hz + 880 Hz) for first half,
# then add 880 Hz tone only in second half
freq1, freq2 = 440, 880

# First half: both frequencies
signal_first = np.sin(2 * np.pi * freq1 * t[:len(t)//2]) + \
               np.sin(2 * np.pi * freq2 * t[:len(t)//2])

# Second half: only freq2 + stronger freq2
signal_second = np.sin(2 * np.pi * freq2 * t[len(t)//2:]) * 1.5

# Combine
audio_signal = np.concatenate([signal_first, signal_second])

# Compute FFT (entire signal)
fft_vals = np.fft.fft(audio_signal)
fft_freq = np.fft.fftfreq(len(audio_signal), 1/sample_rate)
fft_magnitude = np.abs(fft_vals)

# Only positive frequencies
pos_freq_idx = fft_freq > 0
fft_freq_pos = fft_freq[pos_freq_idx]
fft_mag_pos = fft_magnitude[pos_freq_idx]

# Compute spectrogram (STFT)
frequencies, times, Sxx = signal.spectrogram(audio_signal, sample_rate, nperseg=256)

# Visualization
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# Plot 1: Waveform
axes[0].plot(t, audio_signal, color='steelblue', linewidth=0.5)
axes[0].axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Halfway point')
axes[0].set_xlabel('Time (s)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Amplitude', fontsize=11, fontweight='bold')
axes[0].set_title('Audio Waveform: Sum of Sine Waves (440Hz + 880Hz, then 880Hz only)', fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=10)

# Plot 2: FFT (entire signal)
axes[1].plot(fft_freq_pos[:1000], fft_mag_pos[:1000], color='darkred', linewidth=1)
axes[1].axvline(x=freq1, color='green', linestyle='--', linewidth=1.5, label=f'{freq1}Hz')
axes[1].axvline(x=freq2, color='orange', linestyle='--', linewidth=1.5, label=f'{freq2}Hz')
axes[1].set_xlabel('Frequency (Hz)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Magnitude', fontsize=11, fontweight='bold')
axes[1].set_title('FFT: Frequency Components (Entire Signal)', fontsize=12, fontweight='bold')
axes[1].set_xlim([0, 1500])
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=10)

# Plot 3: Spectrogram (time-frequency)
im = axes[2].pcolormesh(times, frequencies, 10 * np.log10(Sxx + 1e-10), shading='gouraud', cmap='viridis')
axes[2].set_ylabel('Frequency (Hz)', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Time (s)', fontsize=11, fontweight='bold')
axes[2].set_title('Spectrogram: Frequency Content Over Time', fontsize=12, fontweight='bold')
axes[2].set_ylim([0, 1500])
axes[2].axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='880Hz increases')
axes[2].legend(fontsize=10)
plt.colorbar(im, ax=axes[2], label='Power (dB)')

plt.tight_layout()
plt.savefig('spectrogram_basics.png', dpi=100, bbox_inches='tight')
plt.show()

print("=== Signal Analysis ===")
print(f"Duration: {duration}s, Sample Rate: {sample_rate}Hz")
print(f"First half: 440Hz + 880Hz sine waves")
print(f"Second half: 880Hz tone (louder)")
print(f"\nFFT shows two peaks at 440Hz and 880Hz.")
print(f"Spectrogram shows how frequency content changes over time.")

## 4. Temporal vs Random Split

Generate time series with trend. Split randomly vs chronologically. Show that random split causes data leakage and misleadingly good test error.

In [ ]:
# Generate time series with trend and cyclical component
n_points = 200
t_series = np.arange(n_points)
trend = 0.5 * t_series
cyclical = 20 * np.sin(2 * np.pi * t_series / 50)
noise = np.random.normal(0, 5, n_points)
y_series = trend + cyclical + noise

# Split 1: Random split (bad for time series!)
np.random.seed(42)
random_indices = np.random.permutation(n_points)
split_point = int(0.7 * n_points)
train_idx_random = random_indices[:split_point]
test_idx_random = random_indices[split_point:]

X_train_random = t_series[train_idx_random].reshape(-1, 1)
y_train_random = y_series[train_idx_random]
X_test_random = t_series[test_idx_random].reshape(-1, 1)
y_test_random = y_series[test_idx_random]

# Split 2: Chronological (correct for time series)
train_idx_chrono = np.arange(split_point)
test_idx_chrono = np.arange(split_point, n_points)

X_train_chrono = t_series[train_idx_chrono].reshape(-1, 1)
y_train_chrono = y_series[train_idx_chrono]
X_test_chrono = t_series[test_idx_chrono].reshape(-1, 1)
y_test_chrono = y_series[test_idx_chrono]

# Fit linear regression to both
model_random = LinearRegression()
model_random.fit(X_train_random, y_train_random)
y_pred_random = model_random.predict(X_test_random)
rmse_random = np.sqrt(mean_squared_error(y_test_random, y_pred_random))

model_chrono = LinearRegression()
model_chrono.fit(X_train_chrono, y_train_chrono)
y_pred_chrono = model_chrono.predict(X_test_chrono)
rmse_chrono = np.sqrt(mean_squared_error(y_test_chrono, y_pred_chrono))

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# Row 1: Random split
axes[0, 0].scatter(t_series[train_idx_random], y_train_random, alpha=0.5, s=30, 
                   color='blue', label='Train', edgecolor='black', linewidth=0.5)
axes[0, 0].scatter(t_series[test_idx_random], y_test_random, alpha=0.5, s=30, 
                   color='red', label='Test', edgecolor='black', linewidth=0.5)
axes[0, 0].plot(t_series, y_series, 'k--', alpha=0.3, linewidth=1, label='True signal')
axes[0, 0].set_xlabel('Time', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Value', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Random Split (Data Leakage!)', fontsize=12, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(alpha=0.3)

# Row 1, Col 2: Predictions (random)
axes[0, 1].plot(t_series, y_series, 'k-', linewidth=1, label='True', alpha=0.7)
# Plot test predictions in order
sorted_test_idx_random = np.sort(test_idx_random)
axes[0, 1].scatter(t_series[sorted_test_idx_random], y_pred_random[np.argsort(test_idx_random)], 
                   alpha=0.6, s=40, color='red', marker='x', linewidth=2, label='Predictions')
axes[0, 1].set_xlabel('Time', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Value', fontsize=11, fontweight='bold')
axes[0, 1].set_title(f'Random Split: RMSE = {rmse_random:.2f} (Misleadingly Good!)', 
                      fontsize=12, fontweight='bold', color='red')
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(alpha=0.3)

# Row 2: Chronological split
axes[1, 0].scatter(t_series[train_idx_chrono], y_train_chrono, alpha=0.5, s=30, 
                   color='blue', label='Train', edgecolor='black', linewidth=0.5)
axes[1, 0].scatter(t_series[test_idx_chrono], y_test_chrono, alpha=0.5, s=30, 
                   color='red', label='Test', edgecolor='black', linewidth=0.5)
axes[1, 0].axvline(x=split_point-0.5, color='green', linestyle='--', linewidth=2, label='Train/Test boundary')
axes[1, 0].plot(t_series, y_series, 'k--', alpha=0.3, linewidth=1, label='True signal')
axes[1, 0].set_xlabel('Time', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Value', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Chronological Split (Correct)', fontsize=12, fontweight='bold', color='green')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(alpha=0.3)

# Row 2, Col 2: Predictions (chrono)
axes[1, 1].plot(t_series, y_series, 'k-', linewidth=1, label='True', alpha=0.7)
axes[1, 1].plot(t_series[test_idx_chrono], y_pred_chrono, 'r-', linewidth=2, label='Predictions')
axes[1, 1].axvline(x=split_point-0.5, color='green', linestyle='--', linewidth=2, label='Forecast boundary')
axes[1, 1].set_xlabel('Time', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Value', fontsize=11, fontweight='bold')
axes[1, 1].set_title(f'Chronological Split: RMSE = {rmse_chrono:.2f}', 
                      fontsize=12, fontweight='bold', color='green')
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('temporal_vs_random_split.png', dpi=100, bbox_inches='tight')
plt.show()

print("=== Temporal Split Analysis ===")
print(f"Random split RMSE: {rmse_random:.4f}")
print(f"Chronological split RMSE: {rmse_chrono:.4f}")
print(f"\nDifference: {rmse_chrono - rmse_random:.4f}")
print(f"\n⚠️ Random split looks unrealistically good!")
print(f"Reason: Training data includes future values, causing data leakage.")
print(f"\nCorrect approach: Train on past, test on future (chronological split).")

## 5. Lag Features for Forecasting

Build lag features (y_{t-1}, y_{t-2}, y_{t-3}) for autoregressive forecasting. Show lag-1 is most important.

In [ ]:
# Generate autoregressive time series: y_t = 0.7*y_{t-1} + noise
n_ar = 300
y_ar = np.zeros(n_ar)
y_ar[0] = np.random.normal(0, 1)
for i in range(1, n_ar):
    y_ar[i] = 0.7 * y_ar[i-1] + np.random.normal(0, 1)

# Build lag features
max_lag = 3
X_lags = np.column_stack([
    y_ar[max_lag-1:-1],      # y_{t-1}
    y_ar[max_lag-2:-2],      # y_{t-2}
    y_ar[max_lag-3:-3],      # y_{t-3}
])
y_target = y_ar[max_lag:]    # y_t (what we predict)

# Chronological split
split = int(0.8 * len(X_lags))
X_lag_train, X_lag_test = X_lags[:split], X_lags[split:]
y_lag_train, y_lag_test = y_target[:split], y_target[split:]

# Train
model_lag = LinearRegression()
model_lag.fit(X_lag_train, y_lag_train)
y_lag_pred = model_lag.predict(X_lag_test)

rmse_lag = np.sqrt(mean_squared_error(y_lag_test, y_lag_pred))

# Inspect coefficients
coef_lags = model_lag.coef_

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# Plot 1: Time series and split
axes[0, 0].plot(y_ar[max_lag:], 'steelblue', linewidth=1, label='Full series')
axes[0, 0].axvline(x=split, color='red', linestyle='--', linewidth=2, label='Train/Test split')
axes[0, 0].fill_between(range(split), min(y_ar), max(y_ar), alpha=0.1, color='blue', label='Train')
axes[0, 0].fill_between(range(split, len(y_ar[max_lag:])), min(y_ar), max(y_ar), alpha=0.1, color='red', label='Test')
axes[0, 0].set_xlabel('Time (t)', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Value', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Autoregressive Series (AR(1): y_t = 0.7*y_{t-1} + noise)', fontsize=11, fontweight='bold')
axes[0, 0].legend(fontsize=9)
axes[0, 0].grid(alpha=0.3)

# Plot 2: Lag feature importance
lag_names = ['y_{t-1}', 'y_{t-2}', 'y_{t-3}']
colors_lags = ['#1f77b4', '#ff7f0e', '#2ca02c']
axes[0, 1].bar(lag_names, np.abs(coef_lags), color=colors_lags, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[0, 1].set_ylabel('|Coefficient|', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Lag Feature Importance', fontsize=12, fontweight='bold')
axes[0, 1].grid(axis='y', alpha=0.3)
for i, (name, coef) in enumerate(zip(lag_names, coef_lags)):
    axes[0, 1].text(i, np.abs(coef), f'{coef:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Plot 3: Actual vs predicted (test set)
test_indices = np.arange(len(y_lag_test))
axes[1, 0].plot(test_indices, y_lag_test, 'k-', linewidth=2, label='Actual', alpha=0.7)
axes[1, 0].plot(test_indices, y_lag_pred, 'r--', linewidth=2, label='Predicted', alpha=0.7)
axes[1, 0].fill_between(test_indices, y_lag_test, y_lag_pred, alpha=0.2, color='gray')
axes[1, 0].set_xlabel('Test Index', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Value', fontsize=11, fontweight='bold')
axes[1, 0].set_title(f'Test Predictions (RMSE = {rmse_lag:.3f})', fontsize=12, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(alpha=0.3)

# Plot 4: Residuals
residuals = y_lag_test - y_lag_pred
axes[1, 1].plot(test_indices, residuals, 'o-', color='purple', linewidth=1, alpha=0.7, markersize=4)
axes[1, 1].axhline(y=0, color='k', linestyle='--', linewidth=1)
axes[1, 1].fill_between(test_indices, residuals, 0, alpha=0.2, color='purple')
axes[1, 1].set_xlabel('Test Index', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Residual', fontsize=11, fontweight='bold')
axes[1, 1].set_title(f'Residuals (mean={np.mean(residuals):.3f}, std={np.std(residuals):.3f})', fontsize=12, fontweight='bold')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('lag_features.png', dpi=100, bbox_inches='tight')
plt.show()

print("=== Lag Features for Forecasting ===")
print(f"Coefficients:")
for name, coef in zip(lag_names, coef_lags):
    print(f"  {name}: {coef:+.4f}")
print(f"\nLag-1 coefficient: {coef_lags[0]:.4f} (close to 0.7, as expected!)")
print(f"Test RMSE: {rmse_lag:.4f}")
print(f"\n** Insight: Lag-1 dominates. Recent history matters most for AR processes. **")

## 6. Concept Drift Simulation

Simulate a time series where the underlying relationship changes at t=250. Show that fixed models degrade over time. Sliding-window models adapt.

In [ ]:
# Generate data with concept drift
n_drift = 500
t_drift = np.arange(n_drift)

# First half: y = 0.5*t + noise
y_drift_first = 0.5 * t_drift[:250] + np.random.normal(0, 5, 250)

# Second half: relationship changes to y = -0.5*t + offset (slope flips!)
y_drift_second = -0.5 * (t_drift[250:] - 250) + 125 + np.random.normal(0, 5, 250)

y_drift = np.concatenate([y_drift_first, y_drift_second])

# Strategy 1: Train on first 200 points, test on rest
X_drift_train = t_drift[:200].reshape(-1, 1)
y_drift_train = y_drift[:200]

model_fixed = LinearRegression()
model_fixed.fit(X_drift_train, y_drift_train)

# Compute rolling accuracy on test set (200-500)
test_start = 200
window_size = 50
rolling_rmse_fixed = []
rolling_rmse_sliding = []

for i in range(test_start, n_drift - window_size):
    # Fixed model
    X_eval = t_drift[i:i+window_size].reshape(-1, 1)
    y_eval = y_drift[i:i+window_size]
    y_pred_fixed = model_fixed.predict(X_eval)
    rmse_fixed = np.sqrt(mean_squared_error(y_eval, y_pred_fixed))
    rolling_rmse_fixed.append(rmse_fixed)
    
    # Sliding window model (retrain on last 50 points)
    if i >= test_start + 50:
        X_train_sliding = t_drift[i-50:i].reshape(-1, 1)
        y_train_sliding = y_drift[i-50:i]
        model_sliding = LinearRegression()
        model_sliding.fit(X_train_sliding, y_train_sliding)
        y_pred_sliding = model_sliding.predict(X_eval)
        rmse_sliding = np.sqrt(mean_squared_error(y_eval, y_pred_sliding))
        rolling_rmse_sliding.append(rmse_sliding)
    else:
        rolling_rmse_sliding.append(np.nan)

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# Plot 1: Time series with drift
axes[0, 0].plot(t_drift, y_drift, 'steelblue', linewidth=1, label='Observed')
axes[0, 0].axvline(x=250, color='red', linestyle='--', linewidth=2, label='Concept drift')
axes[0, 0].fill_between(range(200), min(y_drift), max(y_drift), alpha=0.1, color='green', label='Training data')
axes[0, 0].fill_between(range(200, n_drift), min(y_drift), max(y_drift), alpha=0.1, color='red', label='Test data')
axes[0, 0].set_xlabel('Time (t)', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Value', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Time Series with Concept Drift (slope flips at t=250)', fontsize=12, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(alpha=0.3)

# Plot 2: Predictions (fixed model)
y_pred_full = model_fixed.predict(t_drift.reshape(-1, 1))
axes[0, 1].plot(t_drift, y_drift, 'steelblue', linewidth=1, label='Observed', alpha=0.7)
axes[0, 1].plot(t_drift, y_pred_full, 'r--', linewidth=2, label='Fixed model prediction')
axes[0, 1].axvline(x=250, color='red', linestyle='--', linewidth=1, alpha=0.5)
axes[0, 1].set_xlabel('Time (t)', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Value', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Fixed Model: Degrades After Drift', fontsize=12, fontweight='bold', color='red')
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(alpha=0.3)

# Plot 3: Rolling RMSE comparison
rolling_t = np.arange(len(rolling_rmse_fixed)) + test_start
axes[1, 0].plot(rolling_t, rolling_rmse_fixed, 'r-', linewidth=2, label='Fixed model', marker='o', markersize=4, alpha=0.7)
axes[1, 0].plot(rolling_t[50:], rolling_rmse_sliding[50:], 'g-', linewidth=2, label='Sliding window (retrained)', marker='s', markersize=4, alpha=0.7)
axes[1, 0].axvline(x=250, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Drift point')
axes[1, 0].set_xlabel('Time (t)', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('RMSE (50-point window)', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Rolling RMSE: Fixed vs Adaptive', fontsize=12, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(alpha=0.3)

# Plot 4: RMSE distribution
axes[1, 1].boxplot([rolling_rmse_fixed[:50], rolling_rmse_fixed[50:], rolling_rmse_sliding[50:]],
                   labels=['Fixed\n(before drift)', 'Fixed\n(after drift)', 'Sliding\n(after drift)'],
                   patch_artist=True)
colors_box = ['lightgreen', 'lightcoral', 'lightgreen']
for patch, color in zip(axes[1, 1].artists, colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1, 1].set_ylabel('RMSE', fontsize=11, fontweight='bold')
axes[1, 1].set_title('RMSE Comparison (Sliding Adapts Better)', fontsize=12, fontweight='bold')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('concept_drift.png', dpi=100, bbox_inches='tight')
plt.show()

print("=== Concept Drift Analysis ===")
print(f"Fixed model RMSE (before drift, t=200-250): {np.mean(rolling_rmse_fixed[:50]):.4f}")
print(f"Fixed model RMSE (after drift, t=250-500): {np.mean(rolling_rmse_fixed[50:]):.4f}")
print(f"Degradation: {np.mean(rolling_rmse_fixed[50:]) - np.mean(rolling_rmse_fixed[:50]):.4f}")
print(f"\nSliding window RMSE (after drift, t=250-500): {np.nanmean(rolling_rmse_sliding[50:]):.4f}")
print(f"\n** Insight: Sliding-window (adaptive) models handle drift better. **")
print(f"Use cross-validation with temporal splits to detect and monitor drift.")

## 7. Leakage Detection

Accidentally include a future feature (y_{t+1}) during model training. Show unrealistically high accuracy. Remove it and accuracy drops.

In [ ]:
# Generate simple time series
n_leak = 200
t_leak = np.arange(n_leak)
y_leak = np.sin(2 * np.pi * t_leak / 50) + np.random.normal(0, 0.3, n_leak)

# Build features WITH leakage: y_t and y_{t+1} (the label!)
X_leak_bad = np.column_stack([
    y_leak[:-2],      # y_t (current value)
    y_leak[1:-1],     # y_{t+1} (FUTURE VALUE - LEAKAGE!)
])
y_leak_target = y_leak[2:]  # y_{t+2} (what we predict)

# Build features WITHOUT leakage: only y_t and y_{t-1}
X_leak_good = np.column_stack([
    y_leak[1:-1],     # y_t (current value)
    y_leak[:-2],      # y_{t-1} (past value)
])

# Split
split_leak = int(0.8 * len(X_leak_bad))

# WITH leakage
X_train_bad = X_leak_bad[:split_leak]
y_train_bad = y_leak_target[:split_leak]
X_test_bad = X_leak_bad[split_leak:]
y_test_bad = y_leak_target[split_leak:]

model_bad = LinearRegression()
model_bad.fit(X_train_bad, y_train_bad)
y_pred_bad = model_bad.predict(X_test_bad)
rmse_bad = np.sqrt(mean_squared_error(y_test_bad, y_pred_bad))

# WITHOUT leakage
X_train_good = X_leak_good[:split_leak]
y_train_good = y_leak_target[:split_leak]
X_test_good = X_leak_good[split_leak:]
y_test_good = y_leak_target[split_leak:]

model_good = LinearRegression()
model_good.fit(X_train_good, y_train_good)
y_pred_good = model_good.predict(X_test_good)
rmse_good = np.sqrt(mean_squared_error(y_test_good, y_pred_good))

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# Plot 1: Feature importance WITH leakage
coef_bad = model_bad.coef_
axes[0, 0].bar(['y_t', 'y_{t+1}\n(LEAKAGE!)'], np.abs(coef_bad), color=['skyblue', 'red'], alpha=0.7, edgecolor='black', linewidth=2)
axes[0, 0].set_ylabel('|Coefficient|', fontsize=11, fontweight='bold')
axes[0, 0].set_title('WITH Temporal Leakage', fontsize=12, fontweight='bold', color='red')
axes[0, 0].grid(axis='y', alpha=0.3)
for i, coef in enumerate(coef_bad):
    axes[0, 0].text(i, np.abs(coef), f'{coef:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Plot 2: Feature importance WITHOUT leakage
coef_good = model_good.coef_
axes[0, 1].bar(['y_t', 'y_{t-1}'], np.abs(coef_good), color=['skyblue', 'skyblue'], alpha=0.7, edgecolor='black', linewidth=2)
axes[0, 1].set_ylabel('|Coefficient|', fontsize=11, fontweight='bold')
axes[0, 1].set_title('WITHOUT Leakage (Correct)', fontsize=12, fontweight='bold', color='green')
axes[0, 1].grid(axis='y', alpha=0.3)
for i, coef in enumerate(coef_good):
    axes[0, 1].text(i, np.abs(coef), f'{coef:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Plot 3: Predictions WITH leakage
test_idx_leak = np.arange(len(y_test_bad))
axes[1, 0].plot(test_idx_leak, y_test_bad, 'k-', linewidth=2, label='Actual', alpha=0.7)
axes[1, 0].plot(test_idx_leak, y_pred_bad, 'r--', linewidth=2, label='Predicted', alpha=0.7)
axes[1, 0].fill_between(test_idx_leak, y_test_bad, y_pred_bad, alpha=0.2, color='red')
axes[1, 0].set_xlabel('Test Index', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Value', fontsize=11, fontweight='bold')
axes[1, 0].set_title(f'WITH Leakage: RMSE = {rmse_bad:.4f} (Unrealistically Good!)', 
                      fontsize=12, fontweight='bold', color='red')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(alpha=0.3)

# Plot 4: Predictions WITHOUT leakage
axes[1, 1].plot(test_idx_leak, y_test_good, 'k-', linewidth=2, label='Actual', alpha=0.7)
axes[1, 1].plot(test_idx_leak, y_pred_good, 'g--', linewidth=2, label='Predicted', alpha=0.7)
axes[1, 1].fill_between(test_idx_leak, y_test_good, y_pred_good, alpha=0.2, color='green')
axes[1, 1].set_xlabel('Test Index', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Value', fontsize=11, fontweight='bold')
axes[1, 1].set_title(f'WITHOUT Leakage: RMSE = {rmse_good:.4f} (Realistic)', 
                      fontsize=12, fontweight='bold', color='green')
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('leakage_detection.png', dpi=100, bbox_inches='tight')
plt.show()

print("=== Temporal Leakage Detection ===")
print(f"\nWITH leakage (y_{{t+1}} as feature):")
print(f"  y_t coefficient: {coef_bad[0]:+.4f}")
print(f"  y_{{t+1}} coefficient: {coef_bad[1]:+.4f} ← This is the FUTURE!")
print(f"  Test RMSE: {rmse_bad:.4f}")

print(f"\nWITHOUT leakage (y_{{t-1}} as feature):")
print(f"  y_t coefficient: {coef_good[0]:+.4f}")
print(f"  y_{{t-1}} coefficient: {coef_good[1]:+.4f}")
print(f"  Test RMSE: {rmse_good:.4f}")

print(f"\nRMSE Increase: {rmse_good - rmse_bad:.4f}")
print(f"\n⚠️ Leakage detection checklist:")
print(f"  • Review feature definition: are you using information unavailable at prediction time?")
print(f"  • Check dates: ensure train < test temporally.")
print(f"  • Audit coefficients: suspiciously high weights may indicate leakage.")
print(f"  • Validate in production: real data won't have access to future!")

## What to Notice

1. **Spectrograms reveal temporal dynamics**: FFT shows frequency content across the entire signal; spectrograms show how it changes over time. This is essential for audio and signal analysis.

2. **Chronological splits are mandatory for time series**: Random splits introduce data leakage—your model trains on future data. Always use chronological (temporal) splits: train on past, test on future.

3. **Lag features capture autoregressive structure**: Past values (lags) are often the strongest predictors of future values. Lag-1 usually dominates, but additional lags can capture longer dependencies.

4. **Concept drift degrades fixed models**: When the underlying relationship changes over time (drift), a model trained on historical data loses accuracy. Sliding-window retraining adapts to recent patterns.

5. **Temporal leakage is subtle and catastrophic**: Including features derived from future values (even accidentally) produces unrealistically good test accuracy. At prediction time, you won't have access to the future. Always audit your feature engineering carefully.

**Key Takeaway**: Time series modeling requires careful attention to temporal structure. Use chronological splits, inspect for leakage, monitor for drift, and remember that your model must make predictions in real time using only past and present information.